In [ ]:
!pip install -q wfdb neurokit2 scipy antropy PyWavelets librosa
!pip install -q scikit-learn xgboost lightgbm shap seaborn imbalanced-learn
!pip install -q torchmetrics einops torchaudio statsmodels plotly nolds
!pip install -q transformers   # HuBERT / Wav2Vec2 backbone


In [ ]:
import os, glob, json, time, warnings, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from scipy import signal as sp_signal
from scipy.stats import skew, kurtosis
from scipy.fft import fft, fftfreq
import pywt
import librosa
import antropy as ant

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchmetrics.classification import (MulticlassAccuracy, MulticlassF1Score,
                                          MulticlassAUROC)
from einops import rearrange

from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                      LeaveOneGroupOut)
from sklearn.preprocessing import (StandardScaler, RobustScaler, LabelEncoder)
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, f1_score, balanced_accuracy_score,
                              accuracy_score)
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
import xgboost as xgb
import lightgbm as lgb
import shap
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')
np.random.seed(42); torch.manual_seed(42); random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── GI Motility / Bowel Classes ─────────────────────────────
# 5-class comprehensive GI motility state
GI_CLASSES = {
    0: 'Normal',          # healthy motility, normal bowel sounds
    1: 'Hypomotility',    # reduced/absent BS — ileus, constipation, post-op
    2: 'Hypermotility',   # excessive BS — diarrhoea, early obstruction
    3: 'IBD_Active',      # Crohn's / UC active flare — distinct BS signature
    4: 'Obstruction',     # mechanical obstruction — high-pitched tinkling
}
N_GI_CLASSES  = 5
GI_COLORS     = {
    'Normal':       '#2ecc71',
    'Hypomotility': '#3498db',
    'Hypermotility':'#f39c12',
    'IBD_Active':   '#e74c3c',
    'Obstruction':  '#9b59b6',
}

# ── Sensor configuration (MedVerse vest) ────────────────────
FS_BS      = 8000    # Hz — abdominal acoustic sensor (bowel sounds)
FS_ACC     = 100     # Hz — abdominal accelerometer (motility vibrations)
FS_ECG     = 500     # Hz — ECG (gut-brain axis via HRV)
FS_TEMP    = 1       # Hz — core temp (fever → IBD flare proxy)
SEG_SEC    = 30      # 30-second analysis window (standard BS protocol)

print(f"Device:  {device}")
print(f"Classes: {GI_CLASSES}")
print(f"BS sampling rate: {FS_BS} Hz | Window: {SEG_SEC}s")



In [ ]:
# Validated from WESAD/bowel acoustics literature + IBD clinical data
# Crohn's: mean sound-to-sound interval 1232ms vs 511ms in IBS (PMC 2025)
# IBD smart-shirt study: 281h audio, 24 IBD + 21 healthy (Frontiers 2025)
GI_PHYSIO_PARAMS = {
    0: {  # Normal
        'bs_rate_per_min':  5,      # bowel sounds per minute (clinical: 5-34/min)
        'bs_interval_ms':   450,    # ms between sound events
        'bs_interval_std':  120,
        'bs_duration_ms':   250,    # individual sound duration
        'bs_freq_peak_hz':  300,    # dominant frequency Hz
        'bs_amplitude':     0.35,   # normalised amplitude
        'bs_high_pitch':    False,  # tinkling = obstruction marker
        'acc_rms':          0.008,  # abdominal wall vibration RMS
        'acc_peristalsis':  0.15,   # peristaltic wave amplitude
        'hrv_rmssd_ms':     38,     # HRV: gut-brain axis (vagal tone)
        'hrv_lf_hf':        1.5,
        'core_temp_c':      36.8,
        'label': 'Normal',
    },
    1: {  # Hypomotility (ileus, severe constipation, opioid-induced)
        'bs_rate_per_min':  0.8,    # near-absent bowel sounds
        'bs_interval_ms':   4500,   # prolonged silence
        'bs_interval_std':  800,
        'bs_duration_ms':   120,
        'bs_freq_peak_hz':  180,    # lower frequency
        'bs_amplitude':     0.10,   # very quiet
        'bs_high_pitch':    False,
        'acc_rms':          0.002,  # minimal motility vibration
        'acc_peristalsis':  0.04,
        'hrv_rmssd_ms':     22,     # reduced vagal tone
        'hrv_lf_hf':        2.8,
        'core_temp_c':      36.7,
        'label': 'Hypomotility',
    },
    2: {  # Hypermotility (diarrhoea, early obstruction, anxiety gut)
        'bs_rate_per_min':  38,     # hyperactive
        'bs_interval_ms':   180,    # rapid succession
        'bs_interval_std':  60,
        'bs_duration_ms':   180,
        'bs_freq_peak_hz':  420,    # slightly elevated frequency
        'bs_amplitude':     0.55,   # louder
        'bs_high_pitch':    False,
        'acc_rms':          0.018,
        'acc_peristalsis':  0.30,
        'hrv_rmssd_ms':     28,
        'hrv_lf_hf':        2.2,
        'core_temp_c':      37.1,
        'label': 'Hypermotility',
    },
    3: {  # IBD Active (Crohn's / UC flare)
        # Key finding: Crohn's BS-to-BS interval 1232ms vs 511ms in IBS
        # Distinct prolonged intervals + high-pitched events (PMC 2025)
        'bs_rate_per_min':  12,
        'bs_interval_ms':   1232,   # hallmark extended interval
        'bs_interval_std':  380,
        'bs_duration_ms':   320,    # longer events
        'bs_freq_peak_hz':  580,    # high-pitched tinkling component
        'bs_amplitude':     0.45,
        'bs_high_pitch':    True,   # tinkling in stricturing phenotype
        'acc_rms':          0.012,
        'acc_peristalsis':  0.22,
        'hrv_rmssd_ms':     18,     # significantly reduced (systemic inflammation)
        'hrv_lf_hf':        3.8,    # sympathetic dominance in flare
        'core_temp_c':      37.6,   # low-grade fever common in flare
        'label': 'IBD_Active',
    },
    4: {  # Mechanical Obstruction
        # High-pitched tinkling + rushes → then silence
        'bs_rate_per_min':  22,
        'bs_interval_ms':   511,    # IBS-like interval (early obstruction)
        'bs_interval_std':  200,
        'bs_duration_ms':   400,    # prolonged rush sounds
        'bs_freq_peak_hz':  750,    # high-pitched (clinical hallmark)
        'bs_amplitude':     0.70,   # loud rushes
        'bs_high_pitch':    True,   # CRITICAL marker
        'acc_rms':          0.025,  # strong peristaltic rushes
        'acc_peristalsis':  0.45,
        'hrv_rmssd_ms':     15,     # very low (pain/distress)
        'hrv_lf_hf':        4.5,
        'core_temp_c':      37.2,
        'label': 'Obstruction',
    },
}


In [ ]:
def synthesise_bowel_sound(t, onset_sec, duration_ms, freq_hz,
                             amplitude, high_pitch=False, fs=FS_BS):
    """
    Synthesise a single bowel sound event as a damped sinusoidal burst.
    High-pitched tinkling modelled as harmonic-rich burst (Obstruction/IBD).
    """
    dur_s  = duration_ms / 1000
    t_rel  = t - onset_sec
    mask   = (t_rel >= 0) & (t_rel < dur_s)
    env    = np.exp(-t_rel * 6.0 / (dur_s + 1e-6))  # exponential decay
    env    = np.where(mask, env, 0)

    # Base frequency component
    signal = amplitude * env * np.sin(2*np.pi * freq_hz * t)

    if high_pitch:
        # Add harmonic overtones for tinkling (IBD stricture / obstruction)
        signal += (amplitude * 0.4 * env *
                   np.sin(2*np.pi * freq_hz * 2.3 * t))  # 2nd harmonic
        signal += (amplitude * 0.2 * env *
                   np.sin(2*np.pi * freq_hz * 3.7 * t))  # 3rd harmonic

    # Amplitude modulation (natural variability)
    signal *= (1 + 0.15 * np.sin(2*np.pi * 0.8 * t))
    return signal

def generate_gi_segment(gi_class, subject_id=0,
                          seg_sec=SEG_SEC, seed=None):
    """
    Generate one 30-second multimodal GI monitoring segment.
    Modalities: Bowel Sound audio + Abdominal ACC + ECG + Core Temp
    """
    if seed is not None:
        np.random.seed(seed)

    p = GI_PHYSIO_PARAMS[gi_class]

    # ── Bowel Sound Audio Track ────────────────────────
    n_bs   = int(seg_sec * FS_BS)
    t_bs   = np.arange(n_bs) / FS_BS
    audio  = np.zeros(n_bs, dtype=np.float32)

    # Generate events from Poisson process
    rate_per_sec  = max(0.1, p['bs_rate_per_min'] / 60)
    n_events      = int(np.random.poisson(rate_per_sec * seg_sec))
    n_events      = max(0, min(n_events, int(rate_per_sec * seg_sec * 2)))

    event_times = []
    if n_events > 0:
        # Inter-event intervals from exponential distribution
        iei_mean = p['bs_interval_ms'] / 1000
        iei_std  = p['bs_interval_std'] / 1000
        intervals = np.abs(np.random.normal(iei_mean, iei_std, n_events))
        intervals = np.clip(intervals, 0.05, seg_sec)
        t_cursor  = np.random.uniform(0, 1.0)
        for iei in intervals:
            if t_cursor >= seg_sec - 0.5: break
            event_times.append(t_cursor)
            # Vary freq, amp, duration per event
            freq  = p['bs_freq_peak_hz'] * np.random.uniform(0.85, 1.15)
            amp   = p['bs_amplitude']    * np.random.uniform(0.7,  1.3)
            dur   = p['bs_duration_ms']  * np.random.uniform(0.7,  1.4)
            audio += synthesise_bowel_sound(
                t_bs, t_cursor, dur, freq, amp,
                p['bs_high_pitch'], FS_BS)
            t_cursor += iei

    # Background noise (intestinal gurgling baseline + env noise)
    noise_floor  = 0.015 * np.random.randn(n_bs)
    # Low-freq abdominal background
    bg_gurgle    = (0.02 * np.sin(2*np.pi*80*t_bs) *
                    np.sin(2*np.pi*0.3*t_bs))
    audio       += noise_floor + bg_gurgle
    audio        = np.clip(audio, -1.0, 1.0).astype(np.float32)

    # ── Abdominal Accelerometer ────────────────────────
    n_acc    = int(seg_sec * FS_ACC)
    t_acc    = np.arange(n_acc) / FS_ACC
    acc_base = p['acc_rms'] * np.random.randn(n_acc)

    # Peristaltic waves (0.05–0.15 Hz)
    acc_peri = (p['acc_peristalsis'] *
                np.sin(2*np.pi * np.random.uniform(0.05, 0.15) * t_acc) *
                (1 + 0.3 * np.random.randn(n_acc)))

    # Body movement artefact
    acc_mvt  = 0.003 * np.random.randn(n_acc)
    acc_z    = (acc_base + acc_peri + acc_mvt).astype(np.float32)

    # Transverse axis — minimal
    acc_x    = (0.002 * np.random.randn(n_acc)).astype(np.float32)
    acc_y    = (0.002 * np.random.randn(n_acc)).astype(np.float32)
    acc_3d   = np.stack([acc_x, acc_y, acc_z], axis=1)  # (N, 3)

    # ── ECG (gut-brain axis via HRV) ───────────────────
    n_ecg   = int(seg_sec * FS_ECG)
    hr      = 60000 / max(300, np.random.normal(
        60000 / max(300, p['hrv_rmssd_ms'] * 1.2 + 500), 5))
    hr      = np.clip(hr, 45, 120)
    rr_mean = 60000 / hr
    rr_ints = np.abs(np.random.normal(
        rr_mean, p['hrv_rmssd_ms'] * 2,
        int(seg_sec * hr / 60) + 5))
    rr_ints = np.clip(rr_ints, 300, 1800)

    t_ecg   = np.arange(n_ecg) / FS_ECG
    ecg     = np.zeros(n_ecg, dtype=np.float32)
    t_beat  = 0
    for rr in rr_ints:
        ts = t_beat / 1000
        if ts >= seg_sec: break
        r_idx = int(ts * FS_ECG)
        if r_idx >= n_ecg: break
        for offset, amp, wid in [
            (-0.04, -0.12, 0.008), (0.0, 1.0, 0.005),
            (0.05, -0.18, 0.010)]:
            tl = t_ecg - (ts + offset)
            ecg += amp * np.exp(-tl**2 / (2*wid**2))
        tl = t_ecg - (ts - 0.16)
        ecg += 0.10 * np.exp(-tl**2 / (2*(0.03)**2))
        tl = t_ecg - (ts + 0.30)
        ecg += 0.30 * np.exp(-tl**2 / (2*(0.05)**2))
        t_beat += rr
    ecg += 0.03 * np.random.randn(n_ecg)

    # ── Core Temperature ──────────────────────────────
    n_temp = max(1, int(seg_sec * FS_TEMP))
    temp   = np.random.normal(p['core_temp_c'], 0.15, n_temp).astype(np.float32)

    return {
        'audio':       audio,        # (N_BS,)     — bowel sound
        'acc':         acc_3d,       # (N_ACC, 3)  — abdominal accelerometer
        'ecg':         ecg,          # (N_ECG,)
        'temp':        temp,         # (N_TEMP,)
        'rr_ints':     rr_ints,      # RR interval array
        'event_times': event_times,  # bowel sound onset times (s)
        'n_bs_events': len(event_times),
        'label':       gi_class,
        'subject':     subject_id,
    }

# ── Generate full dataset ──────────────────────────────────
# 25 subjects × 5 classes × 10 segments = 1,250 samples
# Mirrors IBD smart-shirt study (24 IBD + 21 healthy → expanded)
N_SUBJ    = 25
N_SEGS    = 10
print("Generating GI Motility dataset...")
print(f"  {N_SUBJ} subjects × {N_GI_CLASSES} classes × {N_SEGS} segments "
      f"= {N_SUBJ * N_GI_CLASSES * N_SEGS} total")

gi_records = []
for subj in range(N_SUBJ):
    for cls in range(N_GI_CLASSES):
        for seg_i in range(N_SEGS):
            seed = subj * 500 + cls * 100 + seg_i
            rec  = generate_gi_segment(cls, subject_id=subj,
                                        seg_sec=SEG_SEC, seed=seed)
            gi_records.append(rec)

labels_gi   = np.array([r['label'] for r in gi_records])
subjects_gi = np.array([r['subject'] for r in gi_records])
print(f"\nDataset: {len(gi_records)} segments")
for cls, name in GI_CLASSES.items():
    print(f"  {name:15s}: {(labels_gi==cls).sum()} segments")


In [ ]:
fig = plt.figure(figsize=(26, 22))
gs  = gridspec.GridSpec(5, 5, figure=fig, hspace=0.55, wspace=0.38)

class_names = list(GI_CLASSES.values())
sample_recs = {cls: next(r for r in gi_records if r['label']==cls)
               for cls in range(N_GI_CLASSES)}

# ── Row 0: Raw audio waveforms (5s window) ──────────
for col, (cls, rec) in enumerate(sample_recs.items()):
    ax  = fig.add_subplot(gs[0, col])
    n5s = FS_BS * 5
    t   = np.arange(n5s) / FS_BS
    ax.plot(t, rec['audio'][:n5s], lw=0.5,
            color=list(GI_COLORS.values())[cls])
    ev_in_win = [e for e in rec['event_times'] if e < 5]
    for ev in ev_in_win:
        ax.axvline(ev, color='red', lw=0.8, alpha=0.5)
    ax.set_title(f"{GI_CLASSES[cls]}\n"
                 f"{rec['n_bs_events']} events/30s",
                 fontweight='bold', fontsize=9,
                 color=list(GI_COLORS.values())[cls])
    ax.set_xlabel('Time (s)'); ax.set_ylabel('Amplitude')
    ax.grid(True, alpha=0.3)

# ── Row 1: Spectrograms ──────────────────────────────
for col, (cls, rec) in enumerate(sample_recs.items()):
    ax  = fig.add_subplot(gs[1, col])
    f_s, t_s, Sxx = sp_signal.spectrogram(
        rec['audio'], fs=FS_BS,
        nperseg=512, noverlap=384)
    # Show 0–2000 Hz (clinically relevant for BS)
    f_mask = f_s <= 2000
    ax.pcolormesh(t_s, f_s[f_mask],
                   10*np.log10(Sxx[f_mask]+1e-12),
                   shading='gouraud', cmap='inferno')
    ax.axhline(GI_PHYSIO_PARAMS[cls]['bs_freq_peak_hz'],
               color='cyan', lw=1, linestyle='--',
               label=f"Peak {GI_PHYSIO_PARAMS[cls]['bs_freq_peak_hz']}Hz")
    ax.set_title(f'{GI_CLASSES[cls]} Spectrogram',
                  fontweight='bold', fontsize=9)
    ax.set_xlabel('Time (s)'); ax.set_ylabel('Freq (Hz)')
    ax.legend(fontsize=6, loc='upper right')

# ── Row 2: ACC abdominal motility ────────────────────
for col, (cls, rec) in enumerate(sample_recs.items()):
    ax  = fig.add_subplot(gs[2, col])
    t_a = np.arange(len(rec['acc'])) / FS_ACC
    ax.plot(t_a, rec['acc'][:,2], lw=0.8,
            color=list(GI_COLORS.values())[cls], label='ACC-Z (peristaltic)')
    ax.fill_between(t_a, rec['acc'][:,2], alpha=0.15,
                    color=list(GI_COLORS.values())[cls])
    ax.set_title(f'{GI_CLASSES[cls]} Abdominal ACC\n'
                 f"RMS={GI_PHYSIO_PARAMS[cls]['acc_rms']:.4f}",
                 fontweight='bold', fontsize=9)
    ax.set_xlabel('Time (s)'); ax.set_ylabel('g')
    ax.grid(True, alpha=0.3)

# ── Row 3: BS inter-event interval histograms ────────
for col, (cls, rec) in enumerate(sample_recs.items()):
    ax = fig.add_subplot(gs[3, col])
    # Use multiple segments for more data points
    all_iei = []
    for r in gi_records:
        if r['label'] == cls and len(r['event_times']) >= 2:
            iei = np.diff(r['event_times']) * 1000  # ms
            all_iei.extend(iei.tolist())
    if all_iei:
        ax.hist(all_iei, bins=30, color=list(GI_COLORS.values())[cls],
                alpha=0.8, edgecolor='black', density=True)
        ax.axvline(np.mean(all_iei), color='red', lw=1.5,
                   linestyle='--', label=f"μ={np.mean(all_iei):.0f}ms")
    ax.set_title(f'{GI_CLASSES[cls]}\nInter-Event Interval',
                  fontweight='bold', fontsize=9)
    ax.set_xlabel('IEI (ms)'); ax.set_ylabel('Density')
    ax.legend(fontsize=7); ax.grid(True, alpha=0.3)
    # Key annotation: Crohn's 1232ms vs IBS 511ms (PMC 2025)
    if cls == 3:
        ax.text(0.5, 0.85, 'Crohn\'s: 1232ms\nvs IBS: 511ms\n(PMC 2025)',
                transform=ax.transAxes, fontsize=7,
                color='red', fontweight='bold',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# ── Row 4: HRV + Temp comparison ────────────────────
ax_hrv  = fig.add_subplot(gs[4, :3])
ax_temp = fig.add_subplot(gs[4, 3:])

rmssd_by_class = [[GI_PHYSIO_PARAMS[c]['hrv_rmssd_ms'] *
                    np.random.uniform(0.7, 1.3) for _ in range(40)]
                   for c in range(N_GI_CLASSES)]
bp = ax_hrv.boxplot(rmssd_by_class, labels=class_names, patch_artist=True)
for patch, col in zip(bp['boxes'], GI_COLORS.values()):
    patch.set_facecolor(col); patch.set_alpha(0.7)
ax_hrv.axhline(30, color='orange', linestyle='--', lw=1.5,
               label='Stress threshold (30ms)')
ax_hrv.axhline(20, color='red',    linestyle='--', lw=1.5,
               label='Low vagal tone (<20ms)')
ax_hrv.set_title('HRV RMSSD by GI State\n'
                  '(Gut-Brain Axis — vagal tone reflects GI inflammation)',
                  fontweight='bold')
ax_hrv.set_ylabel('RMSSD (ms)'); ax_hrv.legend(fontsize=8)
ax_hrv.grid(True, alpha=0.3)

temps = [GI_PHYSIO_PARAMS[c]['core_temp_c'] for c in range(N_GI_CLASSES)]
bars  = ax_temp.bar(class_names, temps,
                     color=list(GI_COLORS.values()),
                     edgecolor='black', alpha=0.85)
for bar, val in zip(bars, temps):
    ax_temp.text(bar.get_x()+bar.get_width()/2,
                 bar.get_height()+0.01,
                 f'{val:.1f}°C', ha='center', fontsize=9, fontweight='bold')
ax_temp.axhline(37.5, color='red', linestyle='--', lw=1.5,
                label='Fever threshold (37.5°C)')
ax_temp.set_title('Core Temperature by GI State\n(IBD flare → low-grade fever)',
                   fontweight='bold')
ax_temp.set_ylabel('°C'); ax_temp.legend(fontsize=8)
ax_temp.set_ylim(36.3, 38.0); ax_temp.grid(True, alpha=0.3)
ax_temp.tick_params(axis='x', rotation=20)

plt.suptitle("GI Motility — EDA: Acoustic, ACC, HRV & Temperature Signatures",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('gi_motility_eda.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
def extract_bowel_sound_features(audio, fs=FS_BS, seg_sec=SEG_SEC):
    """
    Comprehensive bowel sound feature set.
    Acoustic features: MFCC, spectral, temporal event stats.
    Based on wearable IBD smart-shirt study (Frontiers 2025).
    """
    feats = {}
    audio = np.array(audio, dtype=float)

    # ── Event Detection (energy-threshold) ──────────
    frame_len   = int(fs * 0.025)  # 25ms
    hop         = int(fs * 0.010)  # 10ms
    rms_frames  = np.array([
        np.sqrt(np.mean(audio[i:i+frame_len]**2))
        for i in range(0, len(audio)-frame_len, hop)])

    threshold = np.percentile(rms_frames, 80)
    events    = rms_frames > threshold
    # Count contiguous event blobs
    event_onsets = []
    in_event = False
    for i, e in enumerate(events):
        if e and not in_event:
            event_onsets.append(i * hop / fs)
            in_event = True
        elif not e:
            in_event = False

    n_events  = len(event_onsets)
    feats['bs_event_count']      = float(n_events)
    feats['bs_event_rate']       = float(n_events / seg_sec * 60)  # per min
    feats['bs_energy_ratio']     = float(events.mean())             # fraction active

    # Inter-event intervals
    if n_events >= 2:
        iei_ms = np.diff(event_onsets) * 1000
        feats['bs_iei_mean_ms']  = float(iei_ms.mean())
        feats['bs_iei_std_ms']   = float(iei_ms.std())
        feats['bs_iei_median_ms']= float(np.median(iei_ms))
        feats['bs_iei_cv']       = float(iei_ms.std() / (iei_ms.mean() + 1e-8))
        feats['bs_iei_skew']     = float(skew(iei_ms))
        feats['bs_iei_kurt']     = float(kurtosis(iei_ms))
        # Key biomarker: Crohn's IBD IEI >1000ms (PMC 2025)
        feats['bs_iei_gt1000ms'] = float(np.mean(iei_ms > 1000))
        feats['bs_iei_lt300ms']  = float(np.mean(iei_ms < 300))
    else:
        for k in ['bs_iei_mean_ms','bs_iei_std_ms','bs_iei_median_ms',
                   'bs_iei_cv','bs_iei_skew','bs_iei_kurt',
                   'bs_iei_gt1000ms','bs_iei_lt300ms']:
            feats[k] = 0.0 if 'ms' in k else 1.0 if 'gt' in k else 0.0

    # ── Spectral Features ───────────────────────────
    f_arr, psd = sp_signal.welch(audio, fs=fs, nperseg=min(2048, len(audio)//4))
    total_pwr  = psd.sum() + 1e-12

    # Clinical BS frequency bands
    low_band  = (f_arr >= 20)  & (f_arr < 300)   # low rumbles
    mid_band  = (f_arr >= 300) & (f_arr < 600)   # normal BS
    high_band = (f_arr >= 600) & (f_arr < 1500)  # high-pitched tinkling

    feats['spec_low_power']   = float(psd[low_band].sum()  / total_pwr)
    feats['spec_mid_power']   = float(psd[mid_band].sum()  / total_pwr)
    feats['spec_high_power']  = float(psd[high_band].sum() / total_pwr)
    feats['spec_high_ratio']  = float(psd[high_band].sum() /
                                        (psd[low_band].sum() + 1e-12))
    feats['spec_peak_hz']     = float(f_arr[psd.argmax()])
    feats['spec_centroid']    = float(np.sum(f_arr * psd) / total_pwr)
    feats['spec_bandwidth']   = float(
        np.sqrt(np.sum(((f_arr - feats['spec_centroid'])**2) * psd) / total_pwr))
    feats['spec_rolloff_85']  = float(
        f_arr[np.searchsorted(np.cumsum(psd), 0.85 * total_pwr)])
    feats['spec_flatness']    = float(
        np.exp(np.mean(np.log(psd + 1e-12))) /
        (np.mean(psd) + 1e-12))
    feats['spec_rms']         = float(np.sqrt(np.mean(audio**2)))
    feats['spec_zcr']         = float(np.mean(np.abs(np.diff(np.sign(audio)))) / 2)

    # ── MFCCs (13 coefficients) ──────────────────────
    try:
        mfccs = librosa.feature.mfcc(y=audio.astype(np.float32),
                                      sr=fs, n_mfcc=13,
                                      hop_length=int(fs*0.010),
                                      n_fft=512)
        for i in range(13):
            feats[f'mfcc_{i+1}_mean'] = float(mfccs[i].mean())
            feats[f'mfcc_{i+1}_std']  = float(mfccs[i].std())
        # Delta MFCCs
        dmfccs = librosa.feature.delta(mfccs)
        for i in range(13):
            feats[f'dmfcc_{i+1}_mean'] = float(dmfccs[i].mean())
    except Exception as e:
        for i in range(13):
            feats[f'mfcc_{i+1}_mean'] = feats[f'mfcc_{i+1}_std'] = 0.0
            feats[f'dmfcc_{i+1}_mean'] = 0.0

    # ── Chroma + Spectral Contrast ───────────────────
    try:
        chroma = librosa.feature.chroma_stft(y=audio.astype(np.float32),
                                              sr=fs, n_fft=512)
        feats['chroma_mean'] = float(chroma.mean())
        feats['chroma_std']  = float(chroma.std())
        contrast = librosa.feature.spectral_contrast(
            y=audio.astype(np.float32), sr=fs, n_fft=512)
        feats['spec_contrast_mean'] = float(contrast.mean())
    except:
        feats['chroma_mean'] = feats['chroma_std'] = 0.0
        feats['spec_contrast_mean'] = 0.0

    # ── Nonlinear complexity ─────────────────────────
    try:
        feats['samp_en_audio'] = float(
            ant.sample_entropy(audio[:2000]))
    except: feats['samp_en_audio'] = 0.0
    try:
        feats['perm_en_audio'] = float(
            ant.perm_entropy(audio[:4000], normalize=True))
    except: feats['perm_en_audio'] = 0.0

    return feats

def extract_acc_gi_features(acc_3d, fs=FS_ACC):
    """Abdominal ACC features: peristaltic wave detection, motility index."""
    feats = {}
    acc   = np.array(acc_3d)
    mag   = np.sqrt((acc**2).sum(axis=1))

    feats['acc_rms']      = float(np.sqrt(np.mean(mag**2)))
    feats['acc_std']      = float(mag.std())
    feats['acc_range']    = float(mag.max() - mag.min())
    feats['acc_skew']     = float(skew(mag))
    feats['acc_kurt']     = float(kurtosis(mag))
    feats['acc_z_mean']   = float(acc[:,2].mean())
    feats['acc_z_std']    = float(acc[:,2].std())

    # Peristaltic frequency band (0.05–0.15 Hz)
    f_a, psd_a = sp_signal.welch(mag, fs=fs,
                                   nperseg=min(128, len(mag)//4))
    total_a    = psd_a.sum() + 1e-12
    peri_band  = (f_a >= 0.05) & (f_a < 0.15)
    feats['acc_peristaltic_power'] = float(psd_a[peri_band].sum() / total_a)
    feats['acc_peristaltic_peak']  = float(
        f_a[peri_band][psd_a[peri_band].argmax()] if peri_band.sum() > 0 else 0.0)

    # High-freq tremor band (0.5–5 Hz)
    hf_band = (f_a >= 0.5) & (f_a < 5.0)
    feats['acc_hf_power'] = float(psd_a[hf_band].sum() / total_a)

    # Motility index (energy × frequency content)
    feats['acc_motility_index'] = float(
        feats['acc_rms'] * feats['acc_peristaltic_power'])

    return feats

def extract_hrv_gi_features(rr_ints_ms):
    """Subset of HRV for GI/gut-brain axis monitoring."""
    feats = {}
    rr = np.array(rr_ints_ms, dtype=float)
    rr = rr[(rr > 300) & (rr < 1800)]
    if len(rr) < 4:
        for k in ['hrv_rmssd','hrv_sdnn','hrv_pnn50',
                   'hrv_lf_hf','hrv_mean_hr','hrv_sd1']:
            feats[k] = 0.0
        return feats

    diff_rr = np.diff(rr)
    feats['hrv_rmssd']   = float(np.sqrt(np.mean(diff_rr**2)))
    feats['hrv_sdnn']    = float(np.std(rr, ddof=1))
    feats['hrv_pnn50']   = float(np.mean(np.abs(diff_rr) > 50))
    feats['hrv_mean_hr'] = float(60000 / (rr.mean() + 1e-8))
    feats['hrv_sd1']     = float(np.std(diff_rr) / np.sqrt(2))

    # Frequency domain
    t_rr = np.cumsum(rr) / 1000
    t_u  = np.arange(t_rr[0], t_rr[-1], 0.25)
    if len(t_u) >= 8:
        rr_i = np.interp(t_u, t_rr, rr)
        rr_d = rr_i - np.polyval(np.polyfit(t_u, rr_i, 1), t_u)
        f_r, p_r = sp_signal.welch(rr_d, fs=4.0,
                                    nperseg=min(128, len(rr_d)//2))
        lf   = (f_r >= 0.04) & (f_r < 0.15)
        hf   = (f_r >= 0.15) & (f_r < 0.40)
        feats['hrv_lf_hf'] = float(p_r[lf].sum() / (p_r[hf].sum() + 1e-12))
    else:
        feats['hrv_lf_hf'] = 2.0

    return feats

def extract_all_gi_features(record):
    """Full multimodal GI feature extraction."""
    bs_f    = extract_bowel_sound_features(record['audio'])
    acc_f   = extract_acc_gi_features(record['acc'])
    hrv_f   = extract_hrv_gi_features(record['rr_ints'])
    temp_f  = {
        'temp_mean':     float(record['temp'].mean()),
        'temp_max':      float(record['temp'].max()),
        'temp_fever':    float(record['temp'].max() > 37.5),
    }

    all_f = {}
    all_f.update(bs_f); all_f.update(acc_f)
    all_f.update(hrv_f); all_f.update(temp_f)
    all_f['label']   = record['label']
    all_f['subject'] = record['subject']
    return all_f

print("Extracting multimodal GI features...")
gi_feat_rows = []
for i, rec in enumerate(gi_records):
    gi_feat_rows.append(extract_all_gi_features(rec))
    if (i+1) % 200 == 0:
        print(f"  {i+1}/{len(gi_records)} done")

gi_feat_df = pd.DataFrame(gi_feat_rows).fillna(0)
meta_cols  = ['label','subject']
gi_feat_cols = [c for c in gi_feat_df.columns if c not in meta_cols]

X_gi    = gi_feat_df[gi_feat_cols].values.astype(float)
y_gi    = gi_feat_df['label'].values
grp_gi  = gi_feat_df['subject'].values
print(f"\nFeature matrix: {X_gi.shape} | Features: {len(gi_feat_cols)}")


In [ ]:
logo    = LeaveOneGroupOut()
scaler_gi = RobustScaler()

# ── LOSO cross-validation ────────────────────────────
print("LOSO cross-validation (25-fold, one subject left out)...")
loso_accs_gi, loso_f1s_gi = [], []

for fold_i, (tr_idx, te_idx) in enumerate(logo.split(X_gi, y_gi, grp_gi)):
    X_tr_s = scaler_gi.fit_transform(X_gi[tr_idx])
    X_te_s = scaler_gi.transform(X_gi[te_idx])
    y_tr   = y_gi[tr_idx]; y_te = y_gi[te_idx]

    try:
        k_nn   = min(3, min(np.bincount(y_tr)) - 1)
        sm     = SMOTE(random_state=42, k_neighbors=max(1, k_nn))
        X_tr_s, y_tr = sm.fit_resample(X_tr_s, y_tr)
    except: pass

    lgbm_gi = lgb.LGBMClassifier(n_estimators=200, num_leaves=31,
                                   learning_rate=0.05, class_weight='balanced',
                                   random_state=42, verbose=-1)
    lgbm_gi.fit(X_tr_s, y_tr)
    preds   = lgbm_gi.predict(X_te_s)
    loso_accs_gi.append(accuracy_score(y_te, preds))
    loso_f1s_gi.append(f1_score(y_te, preds, average='macro', zero_division=0))
    if fold_i % 5 == 0:
        print(f"  Subj {fold_i:2d} → Acc={loso_accs_gi[-1]:.3f} | "
              f"F1={loso_f1s_gi[-1]:.3f}")

print(f"\nLOSO → Acc={np.mean(loso_accs_gi):.3f}±{np.std(loso_accs_gi):.3f} | "
      f"F1={np.mean(loso_f1s_gi):.3f}±{np.std(loso_f1s_gi):.3f}")

# ── Standard train/val/test split ─────────────────────
X_tr, X_te, y_tr_f, y_te_f, grp_tr, grp_te = train_test_split(
    X_gi, y_gi, grp_gi, test_size=0.20, stratify=y_gi, random_state=42)
X_tr, X_va, y_tr_f, y_va_f = train_test_split(
    X_tr, y_tr_f, test_size=0.15, stratify=y_tr_f, random_state=42)

scaler_gi = RobustScaler()
X_tr_s    = scaler_gi.fit_transform(X_tr)
X_va_s    = scaler_gi.transform(X_va)
X_te_s    = scaler_gi.transform(X_te)

try:
    sm_gi  = SMOTE(random_state=42, k_neighbors=3)
    X_tr_bal, y_tr_bal = sm_gi.fit_resample(X_tr_s, y_tr_f)
except:
    X_tr_bal, y_tr_bal = X_tr_s, y_tr_f

# ── Classical model suite ─────────────────────────────
print("\nTraining classical GI motility classifiers...")
gi_models = {
    'GradBoost': GradientBoostingClassifier(n_estimators=300, max_depth=5,
                                              learning_rate=0.05,
                                              random_state=42),
    'XGBoost':   xgb.XGBClassifier(n_estimators=300, max_depth=6,
                                    learning_rate=0.05,
                                    use_label_encoder=False,
                                    eval_metric='mlogloss',
                                    random_state=42),
    'LightGBM':  lgb.LGBMClassifier(n_estimators=300, num_leaves=40,
                                      learning_rate=0.05,
                                      class_weight='balanced',
                                      random_state=42, verbose=-1),
    'RF':        RandomForestClassifier(n_estimators=300, max_depth=12,
                                         class_weight='balanced',
                                         random_state=42, n_jobs=-1),
}

gi_results = {}
for name, model in gi_models.items():
    model.fit(X_tr_bal, y_tr_bal)
    preds   = model.predict(X_te_s)
    probas  = model.predict_proba(X_te_s)
    acc     = accuracy_score(y_te_f, preds)
    f1_m    = f1_score(y_te_f, preds, average='macro', zero_division=0)
    bacc    = balanced_accuracy_score(y_te_f, preds)
    try:    auc = roc_auc_score(y_te_f, probas, multi_class='ovr')
    except: auc = 0.5
    gi_results[name] = {'acc':acc,'f1_macro':f1_m,'bacc':bacc,'auc':auc,
                         'preds':preds,'probas':probas}
    print(f"  {name:12s}: Acc={acc:.4f} | F1={f1_m:.4f} | "
          f"BACC={bacc:.4f} | AUC={auc:.4f}")

# SHAP analysis
print("\nSHAP analysis (LightGBM)...")
explainer_gi = shap.TreeExplainer(gi_models['LightGBM'])
shap_gi      = explainer_gi.shap_values(X_te_s[:150])
if isinstance(shap_gi, list):
    shap_gi_mean = np.mean([np.abs(sv) for sv in shap_gi], axis=0).mean(0)
else:
    shap_gi_mean = np.abs(shap_gi).mean(0)
top20_gi  = np.argsort(shap_gi_mean)[-20:]
top20_names_gi = [gi_feat_cols[i] for i in top20_gi]
print("Top 10 GI motility features:")
for f, v in sorted(zip(top20_names_gi, shap_gi_mean[top20_gi]),
                    key=lambda x:-x[1])[:10]:
    print(f"  {f:38s}: {v:.4f}")


In [ ]:
class BowelSoundCNN(nn.Module):
    """
    1D CNN for raw bowel sound classification.
    Inspired by CNN model achieving >98% correlation with manual
    motility labels (PMC 2022) and MRAY multimodal YOLO (2025).
    Input: (B, 1, T) — raw 30s audio
    """
    def __init__(self, n_classes=5, base_ch=32, dropout=0.3):
        super().__init__()

        # Multi-scale temporal convolutions
        self.branch_ms = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(1, base_ch, kernel_size=k, padding=k//2),
                nn.BatchNorm1d(base_ch), nn.GELU(), nn.MaxPool1d(8))
            for k in [11, 31, 101]   # 1.4ms, 3.9ms, 12.6ms at 8kHz
        ])

        # Deeper feature extraction
        self.deep = nn.Sequential(
            nn.Conv1d(base_ch*3, 128, 7, padding=3),
            nn.BatchNorm1d(128), nn.GELU(), nn.MaxPool1d(4),
            nn.Conv1d(128, 256, 5, padding=2),
            nn.BatchNorm1d(256), nn.GELU(), nn.MaxPool1d(4),
            nn.Conv1d(256, 256, 3, padding=1),
            nn.BatchNorm1d(256), nn.GELU(),
            nn.AdaptiveAvgPool1d(16),
        )

        # Temporal attention
        self.attn = nn.Sequential(
            nn.Linear(256, 64), nn.Tanh(),
            nn.Linear(64, 1))

        self.classifier = nn.Sequential(
            nn.Linear(256, 128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, 64),  nn.GELU(), nn.Dropout(dropout/2),
            nn.Linear(64, n_classes)
        )

    def forward(self, x):
        # x: (B, 1, T)
        ms_outs = [branch(x) for branch in self.branch_ms]
        # Align lengths
        min_len = min(o.shape[-1] for o in ms_outs)
        ms_cat  = torch.cat([o[:,:,:min_len] for o in ms_outs], dim=1)
        deep    = self.deep(ms_cat)  # (B, 256, 16)
        # Attention pooling
        a_w     = torch.softmax(self.attn(
            deep.permute(0,2,1)), dim=1)  # (B, 16, 1)
        ctx     = (a_w * deep.permute(0,2,1)).sum(1)  # (B, 256)
        return self.classifier(ctx)

class GIAudioDataset(Dataset):
    """Raw bowel sound audio dataset with augmentation."""
    def __init__(self, records, labels, augment=False, max_len=None):
        self.data   = []
        self.labels = labels
        self.aug    = augment
        self.max_len = max_len or int(SEG_SEC * FS_BS)

        for rec in records:
            a = rec['audio'].astype(np.float32)
            a = a / (np.abs(a).max() + 1e-8)   # normalise
            if len(a) >= self.max_len:
                a = a[:self.max_len]
            else:
                a = np.pad(a, (0, self.max_len - len(a)))
            self.data.append(a)

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        a = self.data[idx].copy()
        if self.aug:
            a *= np.random.uniform(0.7, 1.3)
            a += np.random.randn(len(a)) * 0.008
            shift = np.random.randint(-FS_BS//2, FS_BS//2)
            a     = np.roll(a, shift)
            # Frequency masking (SpecAugment-inspired)
            if np.random.rand() < 0.3:
                mask_start = np.random.randint(0, len(a)//2)
                mask_len   = np.random.randint(FS_BS//4, FS_BS//2)
                a[mask_start:mask_start+mask_len] = 0
        return (torch.FloatTensor(a).unsqueeze(0),
                torch.LongTensor([self.labels[idx]])[0])

# Split by subject
idx_tr_gi = np.where(np.isin(grp_gi, np.unique(grp_tr)))[0]
idx_te_gi = np.where(np.isin(grp_gi, np.unique(grp_te)))[0]

recs_tr_gi = [gi_records[i] for i in idx_tr_gi]
recs_te_gi = [gi_records[i] for i in idx_te_gi]
lbl_tr_gi  = y_gi[idx_tr_gi].tolist()
lbl_te_gi  = y_gi[idx_te_gi].tolist()

audio_tr_ds = GIAudioDataset(recs_tr_gi, lbl_tr_gi, augment=True)
audio_te_ds = GIAudioDataset(recs_te_gi, lbl_te_gi, augment=False)
audio_tr_dl = DataLoader(audio_tr_ds, batch_size=16, shuffle=True,  num_workers=2)
audio_te_dl = DataLoader(audio_te_ds, batch_size=16, shuffle=False, num_workers=2)

bs_cnn = BowelSoundCNN(n_classes=N_GI_CLASSES).to(device)
print(f"BowelSoundCNN | Params: {sum(p.numel() for p in bs_cnn.parameters()):,}")
print(f"Train: {len(audio_tr_ds)} | Test: {len(audio_te_ds)} audio windows")


In [ ]:
class GIPatchTransformer(nn.Module):
    """
    Patch-based transformer for bowel sound classification.
    Mirrors HuBERT/Wav2Vec2 architecture achieving
    F1 80-85% on IBD detection (PMC 2025).
    Input: (B, 1, T) raw audio
    """
    def __init__(self, n_classes=5, patch_sec=0.5,
                 d_model=128, n_heads=4, n_layers=4, dropout=0.2):
        super().__init__()

        patch_len = int(patch_sec * FS_BS)   # 0.5s = 4000 samples
        n_patches = int(SEG_SEC / patch_sec)  # 60 patches

        # Patch embedding via strided conv
        self.patch_embed = nn.Sequential(
            nn.Conv1d(1, 64,  kernel_size=80, stride=40, padding=40),
            nn.GELU(),
            nn.Conv1d(64, d_model, kernel_size=5, stride=4, padding=2),
            nn.GELU(),
            nn.AdaptiveAvgPool1d(n_patches),
        )

        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pos_embed = nn.Parameter(
            torch.randn(1, n_patches + 1, d_model) * 0.02)
        self.drop_emb  = nn.Dropout(dropout)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=d_model*4, dropout=dropout,
            activation='gelu', batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(enc_layer,
                                                  num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(
            nn.Linear(d_model, 64), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, n_classes)
        )

    def forward(self, x):
        h   = self.patch_embed(x)              # (B, d_model, P)
        h   = h.permute(0, 2, 1)              # (B, P, d_model)
        B   = h.shape[0]
        cls = self.cls_token.expand(B, -1, -1)
        h   = torch.cat([cls, h], dim=1)
        h   = h + self.pos_embed[:, :h.shape[1]]
        h   = self.drop_emb(h)
        out = self.transformer(h)
        out = self.norm(out[:, 0])
        return self.head(out)

gi_transformer = GIPatchTransformer(n_classes=N_GI_CLASSES).to(device)
print(f"GIPatchTransformer | Params: {sum(p.numel() for p in gi_transformer.parameters()):,}")


In [ ]:
class GIMultimodalNet(nn.Module):
    """
    Full 4-modality GI motility classifier:
    Bowel Sound Audio + Abdominal ACC + ECG + Temperature
    Feature-level cross-modal fusion with gating.
    """
    def __init__(self, n_classes=5, embed=64, dropout=0.3):
        super().__init__()

        # Audio encoder (compressed)
        self.audio_enc = nn.Sequential(
            nn.Conv1d(1, 32, 15, stride=4, padding=7), nn.GELU(), nn.MaxPool1d(4),
            nn.Conv1d(32, 64, 7, stride=2, padding=3), nn.GELU(), nn.MaxPool1d(4),
            nn.Conv1d(64, embed, 5, padding=2), nn.GELU(),
            nn.AdaptiveAvgPool1d(8),
            nn.Flatten(), nn.Linear(embed*8, embed), nn.LayerNorm(embed))

        # ACC encoder
        self.acc_enc = nn.Sequential(
            nn.Conv1d(3, 16, 7, padding=3), nn.GELU(), nn.MaxPool1d(4),
            nn.Conv1d(16, 32, 5, padding=2), nn.GELU(), nn.MaxPool1d(2),
            nn.Conv1d(32, embed, 3, padding=1), nn.GELU(),
            nn.AdaptiveAvgPool1d(4),
            nn.Flatten(), nn.Linear(embed*4, embed), nn.LayerNorm(embed))

        # ECG encoder
        self.ecg_enc = nn.Sequential(
            nn.Conv1d(1, 16, 11, padding=5), nn.GELU(), nn.MaxPool1d(4),
            nn.Conv1d(16, 32, 7, padding=3), nn.GELU(), nn.MaxPool1d(4),
            nn.Conv1d(32, embed, 5, padding=2), nn.GELU(),
            nn.AdaptiveAvgPool1d(4),
            nn.Flatten(), nn.Linear(embed*4, embed), nn.LayerNorm(embed))

        # Temperature encoder (scalar → embed)
        self.temp_enc = nn.Sequential(
            nn.Linear(1, 16), nn.GELU(),
            nn.Linear(16, embed), nn.LayerNorm(embed))

        self.n_mod = 4

        # Cross-modal attention
        self.cross_attn = nn.MultiheadAttention(embed, num_heads=4,
                                                  dropout=dropout,
                                                  batch_first=True)
        self.cross_norm = nn.LayerNorm(embed)

        # Gating: learn to suppress noisy modalities
        self.gate = nn.Sequential(
            nn.Linear(embed * self.n_mod, self.n_mod),
            nn.Sigmoid())

        self.classifier = nn.Sequential(
            nn.Linear(embed * self.n_mod, 256),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, 64), nn.GELU(),
            nn.Dropout(dropout/2),
            nn.Linear(64, n_classes))

    def forward(self, audio, acc, ecg, temp,
                 return_gates=False):
        e_a = self.audio_enc(audio.unsqueeze(1) if audio.dim()==2 else audio)
        e_c = self.acc_enc(acc.permute(0,2,1) if acc.dim()==3 else acc)
        e_e = self.ecg_enc(ecg.unsqueeze(1)   if ecg.dim()==2 else ecg)
        t_mean = temp.mean(dim=1, keepdim=True) if temp.dim()==2 else temp.unsqueeze(1)
        e_t = self.temp_enc(t_mean)

        modal = torch.stack([e_a, e_c, e_e, e_t], dim=1)  # (B, 4, D)
        attn_out, _ = self.cross_attn(modal, modal, modal)
        modal = self.cross_norm(attn_out + modal)

        fused = modal.flatten(1)                   # (B, 4D)
        gates = self.gate(fused)                   # (B, 4)
        fused_g = (modal * gates.unsqueeze(-1)).flatten(1)

        logits = self.classifier(fused_g)
        if return_gates:
            return logits, gates.detach().cpu()
        return logits

gi_mm_net = GIMultimodalNet(n_classes=N_GI_CLASSES).to(device)
print(f"GIMultimodalNet | Params: {sum(p.numel() for p in gi_mm_net.parameters()):,}")


In [ ]:
class GIMultimodalDataset(Dataset):
    """Full 4-modality GI dataset."""
    def __init__(self, records, labels, augment=False):
        self.data   = []; self.labels = labels; self.aug = augment
        audio_len = int(SEG_SEC * FS_BS)
        acc_len   = int(SEG_SEC * FS_ACC)
        ecg_len   = int(SEG_SEC * FS_ECG)
        temp_len  = max(1, int(SEG_SEC * FS_TEMP))

        for rec in records:
            def n(sig):
                s = np.array(sig, dtype=np.float32)
                return (s - s.mean()) / (s.std() + 1e-8)

            audio = n(rec['audio'])[:audio_len]
            audio = np.pad(audio, (0, max(0, audio_len-len(audio))))

            acc = rec['acc'][:acc_len].astype(np.float32)
            acc = np.pad(acc, ((0, max(0, acc_len-len(acc))), (0,0)))

            ecg = n(rec['ecg'])[:ecg_len]
            ecg = np.pad(ecg, (0, max(0, ecg_len-len(ecg))))

            temp = rec['temp'][:temp_len].astype(np.float32)
            temp = np.pad(temp, (0, max(0, temp_len-len(temp))))

            self.data.append((audio, acc, ecg, temp))

    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        audio, acc, ecg, temp = self.data[idx]
        if self.aug and np.random.rand() < 0.5:
            audio *= np.random.uniform(0.85, 1.15)
            audio += np.random.randn(len(audio)) * 0.01
        return (torch.FloatTensor(audio),
                torch.FloatTensor(acc),
                torch.FloatTensor(ecg),
                torch.FloatTensor(temp),
                torch.LongTensor([self.labels[idx]])[0])

gi_mm_tr_ds = GIMultimodalDataset(recs_tr_gi, lbl_tr_gi, augment=True)
gi_mm_te_ds = GIMultimodalDataset(recs_te_gi, lbl_te_gi)
gi_mm_tr_dl = DataLoader(gi_mm_tr_ds, batch_size=16, shuffle=True,  num_workers=2)
gi_mm_te_dl = DataLoader(gi_mm_te_ds, batch_size=16, shuffle=False)
gi_mm_va_dl = DataLoader(GIMultimodalDataset(recs_te_gi[:20],
                          lbl_te_gi[:20]), batch_size=8, shuffle=False)

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None):
        super().__init__()
        self.gamma = gamma; self.alpha = alpha
    def forward(self, inp, tgt):
        ce = F.cross_entropy(inp, tgt, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce)
        return ((1-pt)**self.gamma * ce).mean()

def train_gi_model(model, name, tr_dl, va_dl,
                    epochs=80, lr=1e-3, model_type='audio'):
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        opt, T_0=20, eta_min=1e-6)
    cw    = torch.FloatTensor([
        len(y_gi) / (N_GI_CLASSES * (y_gi==c).sum())
        for c in range(N_GI_CLASSES)]).to(device)
    criterion = FocalLoss(gamma=2.0, alpha=cw)

    hist = {'train_loss':[],'val_acc':[],'val_f1':[]}
    best_f1 = 0; patience_ctr = 0; patience = 15

    for epoch in range(epochs):
        model.train()
        losses = []
        for batch in tr_dl:
            if model_type == 'multimodal':
                audio, acc, ecg, temp, lbl = [b.to(device) for b in batch]
                logits = model(audio, acc, ecg, temp)
            else:
                sig, lbl = batch
                sig = sig.to(device); lbl = lbl.to(device)
                logits = model(sig)
            loss = criterion(logits, lbl)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            losses.append(loss.item())

        model.eval()
        preds_v, trues_v = [], []
        with torch.no_grad():
            for batch in va_dl:
                if model_type == 'multimodal':
                    audio, acc, ecg, temp, lbl = [b.to(device) for b in batch]
                    logits = model(audio, acc, ecg, temp)
                else:
                    sig, lbl = batch
                    sig = sig.to(device)
                    logits = model(sig)
                preds_v.extend(logits.argmax(1).cpu().numpy())
                trues_v.extend(lbl.cpu().numpy())

        val_acc = accuracy_score(trues_v, preds_v)
        val_f1  = f1_score(trues_v, preds_v, average='macro', zero_division=0)
        hist['train_loss'].append(np.mean(losses))
        hist['val_acc'].append(val_acc)
        hist['val_f1'].append(val_f1)
        sched.step()

        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), f'best_{name}.pt')
            patience_ctr = 0
        else:
            patience_ctr += 1

        if (epoch+1) % 20 == 0:
            print(f"[{name}] Ep {epoch+1:3d} | "
                  f"Loss:{np.mean(losses):.4f} | "
                  f"Acc:{val_acc:.4f} | F1:{val_f1:.4f}")
        if patience_ctr >= patience:
            print(f"  Early stop @ {epoch+1}"); break

    model.load_state_dict(torch.load(f'best_{name}.pt'))
    print(f"\n{name}: Best F1={best_f1:.4f}")
    return model, hist

# Build val loader for audio models
gi_va_ds = GIAudioDataset(recs_te_gi[:20], lbl_te_gi[:20])
gi_va_dl = DataLoader(gi_va_ds, batch_size=8, shuffle=False)

print("="*65)
print("Training BowelSoundCNN (raw audio)")
print("="*65)
bs_cnn, hist_bs_cnn = train_gi_model(
    bs_cnn, 'gi_bs_cnn', audio_tr_dl, gi_va_dl,
    epochs=80, lr=1e-3, model_type='audio')[:2]

print("\n" + "="*65)
print("Training GIPatchTransformer (Wav2Vec2-style)")
print("="*65)
gi_transformer, hist_gi_trans = train_gi_model(
    gi_transformer, 'gi_transformer', audio_tr_dl, gi_va_dl,
    epochs=80, lr=3e-4, model_type='audio')[:2]

print("\n" + "="*65)
print("Training GIMultimodalNet (Audio+ACC+ECG+Temp)")
print("="*65)
gi_mm_net, hist_gi_mm = train_gi_model(
    gi_mm_net, 'gi_mm', gi_mm_tr_dl, gi_mm_va_dl,
    epochs=80, lr=5e-4, model_type='multimodal')[:2]


In [ ]:
def eval_gi_model(model, dl, model_type='audio'):
    model.eval()
    preds, probas, trues = [], [], []
    with torch.no_grad():
        for batch in dl:
            if model_type == 'multimodal':
                audio, acc, ecg, temp, lbl = [b.to(device) for b in batch]
                logits = model(audio, acc, ecg, temp)
            else:
                sig, lbl = batch
                logits = model(sig.to(device))
            p = F.softmax(logits, dim=1).cpu().numpy()
            preds.extend(logits.argmax(1).cpu().numpy())
            probas.extend(p)
            trues.extend(lbl.cpu().numpy())
    return np.array(preds), np.array(probas), np.array(trues)

fig, axes = plt.subplots(3, 4, figsize=(28, 20))
cn         = [GI_CLASSES[i] for i in range(N_GI_CLASSES)]
dl_gi_res  = {}

for (model, name, mtype, dl), ax in zip([
    (bs_cnn,        'BS-CNN',      'audio',     audio_te_dl),
    (gi_transformer,'Transformer', 'audio',     audio_te_dl),
    (gi_mm_net,     'Multimodal',  'multimodal',gi_mm_te_dl),
], axes[0]):
    p, pr, t = eval_gi_model(model, dl, mtype)
    acc  = accuracy_score(t, p)
    f1   = f1_score(t, p, average='macro', zero_division=0)
    bacc = balanced_accuracy_score(t, p)
    try:    auc = roc_auc_score(t, pr, multi_class='ovr')
    except: auc = 0.5
    dl_gi_res[name] = {'acc':acc,'f1':f1,'bacc':bacc,'auc':auc,
                        'preds':p,'probas':pr,'trues':t}
    cm = confusion_matrix(t, p)
    sns.heatmap(cm, annot=True, fmt='d', ax=ax,
                xticklabels=[c[:7] for c in cn],
                yticklabels=[c[:7] for c in cn],
                cmap='Blues', linewidths=0.5)
    ax.set_title(f'{name}\nAcc={acc:.3f} F1={f1:.3f}',
                  fontweight='bold', fontsize=9)
    ax.tick_params(axis='x', rotation=25, labelsize=7)
    ax.tick_params(axis='y', rotation=0,  labelsize=7)

# Best classical
best_cls_gi = max(gi_results, key=lambda k: gi_results[k]['f1_macro'])
cm_cls = confusion_matrix(y_te_f, gi_results[best_cls_gi]['preds'])
sns.heatmap(cm_cls, annot=True, fmt='d', ax=axes[0,3],
            xticklabels=[c[:7] for c in cn],
            yticklabels=[c[:7] for c in cn],
            cmap='Oranges', linewidths=0.5)
axes[0,3].set_title(f'{best_cls_gi}\nAcc={gi_results[best_cls_gi]["acc"]:.3f} '
                     f'F1={gi_results[best_cls_gi]["f1_macro"]:.3f}',
                     fontweight='bold', fontsize=9)
axes[0,3].tick_params(axis='x', rotation=25, labelsize=7)

# ── Training curves ────────────────────────────────
for hist, name, color in [
    (hist_bs_cnn,   'BS-CNN',       '#e74c3c'),
    (hist_gi_trans, 'Transformer',  '#3498db'),
    (hist_gi_mm,    'Multimodal',   '#2ecc71'),
]:
    axes[1,0].plot(hist['val_acc'], lw=1.8, color=color, label=name)
    axes[1,0].plot(hist['val_f1'],  lw=1.0, color=color, linestyle='--', alpha=0.7)
axes[1,0].axhline(0.83, color='gray', linestyle='--', lw=1,
                   label='Transformer SOTA F1=0.83')
axes[1,0].set_title('Val Acc (—) & F1 (- -)\n', fontweight='bold')
axes[1,0].set_xlabel('Epoch'); axes[1,0].set_ylabel('Score')
axes[1,0].legend(fontsize=8); axes[1,0].grid(True, alpha=0.3)

# ── Model comparison ──────────────────────────────
all_accs_gi = {
    **{k: v['acc'] for k,v in gi_results.items()},
    **{k: v['acc'] for k,v in dl_gi_res.items()},
    'Wav2Vec\n(lit F1≈0.85)': 0.830,
    'IBD Smart\nT-Shirt (lit)': 0.820,
    'HRM AI\n(lit 97%)': 0.970,
}
bar_c = (['#f39c12']*4 + ['#3498db','#e74c3c','#2ecc71'] + ['#95a5a6']*3)
bars  = axes[1,1].bar(range(len(all_accs_gi)),
                       list(all_accs_gi.values()),
                       color=bar_c[:len(all_accs_gi)],
                       edgecolor='black', alpha=0.85)
for bar, val in zip(bars, all_accs_gi.values()):
    axes[1,1].text(bar.get_x()+bar.get_width()/2,
                   bar.get_height()+0.008,
                   f'{val:.3f}', ha='center', fontsize=7, fontweight='bold')
axes[1,1].set_xticks(range(len(all_accs_gi)))
axes[1,1].set_xticklabels(list(all_accs_gi.keys()),
                            rotation=35, ha='right', fontsize=7)
axes[1,1].set_title('Accuracy — All Models vs Literature',
                     fontweight='bold')
axes[1,1].set_ylabel('Accuracy'); axes[1,1].set_ylim(0.3, 1.1)
axes[1,1].grid(True, alpha=0.3)

# ── LOSO per subject ──────────────────────────────
axes[1,2].bar(range(N_SUBJ), loso_accs_gi,
               color='#16a085', edgecolor='black', alpha=0.8)
axes[1,2].axhline(np.mean(loso_accs_gi), color='red',
                   linestyle='--', lw=2,
                   label=f'Mean={np.mean(loso_accs_gi):.3f}')
axes[1,2].set_title('LOSO Accuracy per Subject\n(Subject-Independent)',
                     fontweight='bold')
axes[1,2].set_xlabel('Subject'); axes[1,2].set_ylabel('Accuracy')
axes[1,2].legend(fontsize=9); axes[1,2].grid(True, alpha=0.3)
axes[1,2].set_ylim(0, 1.1)

# ── SHAP feature importance ────────────────────────
top15_gi = sorted(zip(top20_names_gi, shap_gi_mean[top20_gi]),
                   key=lambda x: -x[1])[:15]
s_names, s_vals = zip(*top15_gi)
axes[1,3].barh(range(len(s_names)),
                list(reversed(s_vals)),
                color='#9b59b6', alpha=0.8, edgecolor='black')
axes[1,3].set_yticks(range(len(s_names)))
axes[1,3].set_yticklabels([n[:30] for n in reversed(s_names)], fontsize=7)
axes[1,3].set_title('Top-15 GI Features (SHAP)', fontweight='bold')
axes[1,3].set_xlabel('Mean |SHAP|'); axes[1,3].grid(True, alpha=0.3)

# ── IEI distribution comparison ───────────────────
for cls in range(N_GI_CLASSES):
    iei_all = []
    for r in gi_records:
        if r['label'] == cls and len(r['event_times']) >= 2:
            iei_all.extend(np.diff(r['event_times']) * 1000)
    if iei_all:
        axes[2,0].hist(iei_all, bins=30, alpha=0.55,
                       color=list(GI_COLORS.values())[cls],
                       label=GI_CLASSES[cls], density=True)
axes[2,0].axvline(511,  color='blue',   lw=2, linestyle='--',
                   label='IBS: 511ms')
axes[2,0].axvline(1232, color='red',    lw=2, linestyle='--',
                   label="Crohn's: 1232ms")
axes[2,0].set_title('IEI Distribution by GI State\n'
                     '(PMC 2025 hallmark difference)',
                     fontweight='bold')
axes[2,0].set_xlabel('Inter-Event Interval (ms)')
axes[2,0].set_ylabel('Density')
axes[2,0].legend(fontsize=6); axes[2,0].grid(True, alpha=0.3)
axes[2,0].set_xlim(0, 5000)

# ── Spectral power by class ────────────────────────
spec_data = {'Low (20-300Hz)':[], 'Mid (300-600Hz)':[],
             'High (600-1500Hz)':[], 'Class':[]}
for rec in gi_records[:200]:
    bf = extract_bowel_sound_features(rec['audio'])
    spec_data['Low (20-300Hz)'].append(bf['spec_low_power'])
    spec_data['Mid (300-600Hz)'].append(bf['spec_mid_power'])
    spec_data['High (600-1500Hz)'].append(bf['spec_high_power'])
    spec_data['Class'].append(GI_CLASSES[rec['label']])

spec_df = pd.DataFrame(spec_data)
x_pos   = np.arange(N_GI_CLASSES)
w       = 0.25
for i, (band, col) in enumerate(zip(
    ['Low (20-300Hz)','Mid (300-600Hz)','High (600-1500Hz)'],
    ['#3498db','#2ecc71','#e74c3c'])):
    means = [spec_df[spec_df['Class']==c][band].mean()
             for c in class_names]
    axes[2,1].bar(x_pos + i*w, means, w, label=band,
                   color=col, alpha=0.8, edgecolor='black')
axes[2,1].set_xticks(x_pos + w)
axes[2,1].set_xticklabels([c[:10] for c in class_names],
                             rotation=20, fontsize=8)
axes[2,1].set_title('Spectral Power Bands by GI State\n'
                     '(High freq → obstruction marker)',
                     fontweight='bold')
axes[2,1].set_ylabel('Relative Power')
axes[2,1].legend(fontsize=7); axes[2,1].grid(True, alpha=0.3)

# ── Modality gate weights (GIMultimodalNet) ────────
gi_mm_net.eval()
gate_weights = []
with torch.no_grad():
    for batch in gi_mm_te_dl:
        audio, acc, ecg, temp, lbl = [b.to(device) for b in batch]
        _, gates = gi_mm_net(audio, acc, ecg, temp, return_gates=True)
        gate_weights.append(gates.numpy())
gate_weights = np.vstack(gate_weights)
gate_means   = gate_weights.mean(0)

axes[2,2].bar(['Audio\n(BS)', 'ACC\n(abdom)', 'ECG\n(HRV)', 'Temp'],
               gate_means,
               color=['#e74c3c','#3498db','#2ecc71','#f39c12'],
               edgecolor='black', alpha=0.85)
for i, v in enumerate(gate_means):
    axes[2,2].text(i, v+0.01, f'{v:.3f}', ha='center',
                   fontsize=10, fontweight='bold')
axes[2,2].set_title('Modality Gate Weights\n(GIMultimodalNet — learned importance)',
                     fontweight='bold')
axes[2,2].set_ylabel('Gate Value (0–1)')
axes[2,2].set_ylim(0, 1.15); axes[2,2].grid(True, alpha=0.3)

# ── GI Motility Index (continuous score) ──────────
gi_motility_scores = []
for rec in gi_records:
    bf  = extract_bowel_sound_features(rec['audio'])
    af  = extract_acc_gi_features(rec['acc'])
    score = float(np.clip(
        0.35 * min(bf['bs_event_rate'] / 34.0, 1.0) +
        0.25 * min(bf['spec_mid_power'] * 3, 1.0) +
        0.20 * min(af['acc_motility_index'] * 50, 1.0) +
        0.20 * min(bf['spec_high_power'] * 5, 1.0),
        0, 1))
    gi_motility_scores.append({'score': score,
                                 'label': rec['label']})

gm_df = pd.DataFrame(gi_motility_scores)
vp    = axes[2,3].violinplot(
    [gm_df[gm_df['label']==c]['score'].values
     for c in range(N_GI_CLASSES)],
    positions=range(N_GI_CLASSES),
    showmedians=True, showmeans=True)
for pc, col in zip(vp['bodies'], GI_COLORS.values()):
    pc.set_facecolor(col); pc.set_alpha(0.6)
axes[2,3].set_xticks(range(N_GI_CLASSES))
axes[2,3].set_xticklabels([GI_CLASSES[c][:10] for c in range(N_GI_CLASSES)],
                            rotation=20, fontsize=8)
axes[2,3].axhline(0.3, color='green',  linestyle='--', lw=1,
                   label='Hypo threshold')
axes[2,3].axhline(0.7, color='orange', linestyle='--', lw=1,
                   label='Hyper threshold')
axes[2,3].set_title('GI Motility Index by Class\n(Composite continuous score)',
                     fontweight='bold')
axes[2,3].set_ylabel('Motility Index (0=absent → 1=hyperactive)')
axes[2,3].legend(fontsize=7); axes[2,3].grid(True, alpha=0.3)

plt.suptitle("GI Motility / Bowel Classifier — Complete Evaluation",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('gi_motility_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
import pickle

torch.save(bs_cnn.state_dict(),         'medverse_gi_bs_cnn.pt')
torch.save(gi_transformer.state_dict(), 'medverse_gi_transformer.pt')
torch.save(gi_mm_net.state_dict(),      'medverse_gi_multimodal.pt')
with open('medverse_gi_lgbm.pkl', 'wb') as f:
    pickle.dump(gi_models['LightGBM'], f)
with open('medverse_gi_scaler.pkl', 'wb') as f:
    pickle.dump(scaler_gi, f)
with open('medverse_gi_feat_cols.pkl', 'wb') as f:
    pickle.dump(gi_feat_cols, f)

config_gi = {
    'task': 'gi_motility_bowel_classification',
    'n_classes': 5,
    'classes': GI_CLASSES,
    'gi_motility_index': {
        '0.0–0.3': 'Hypomotility — absent/rare BS, minimal peristalsis',
        '0.3–0.7': 'Normal GI motility',
        '0.7–1.0': 'Hypermotility / active bowel disease',
    },
    'datasets': {
        'primary_ref':   'IBD Smart T-Shirt — 24 IBD + 21 healthy, 281h audio (Frontiers 2025)',
        'acoustic_ref':  'Transformer BS analytics — F1 80–85% IBD (PMC 2025)',
        'hrm_ref':       'AI HRM interpretation — 97% accuracy (JMIR 2025)',
        'cnn_bs_ref':    'CNN bowel sound motility — >98% correlation (PMC 2022)',
        'ibd_biomarker': 'Crohn\'s IEI=1232ms vs IBS=511ms (PMC 2025)',
    },
    'models': {
        'bs_cnn':       'medverse_gi_bs_cnn.pt        (multi-scale raw audio CNN)',
        'transformer':  'medverse_gi_transformer.pt   (Wav2Vec2-style patch transformer)',
        'multimodal':   'medverse_gi_multimodal.pt    (Audio+ACC+ECG+Temp fusion)',
        'lgbm':         'medverse_gi_lgbm.pkl         (acoustic+HRV features)',
        'scaler':       'medverse_gi_scaler.pkl',
        'feat_cols':    'medverse_gi_feat_cols.pkl',
    },
    'hardware': {
        'vest_sensors': {
            'Bowel Sound':   'MEMS microphone embedded in abdominal patch @ 8kHz',
            'Abdominal ACC': '3-axis MEMS accelerometer on abdomen @ 100Hz',
            'ECG':           '3-lead chest @ 500Hz (HRV for gut-brain axis)',
            'Temperature':   'NTC thermistor — core temp proxy @ 1Hz',
        },
        'sensor_placement': {
            'Microphone':    'Left iliac fossa (dominant BS region)',
            'ACC':           'Periumbilical — maximum peristaltic signal',
            'ECG':           'Standard chest placement',
        },
        'window_sec':         30,
        'stride_sec':         15,
        'min_recording_sec':  120,
    },
    'key_biomarkers': {
        'bs_iei_mean_ms':     'Inter-event interval — Crohn\'s >1000ms hallmark',
        'bs_event_rate':      'Events/min — hypomotility <2, hypermotility >30',
        'spec_high_power':    'High-freq power >600Hz — obstruction/IBD marker',
        'acc_motility_index': 'Peristaltic wave energy composite',
        'hrv_rmssd':          'Vagal tone — reduced in IBD inflammation',
        'temp_fever':         'Binary fever flag — IBD flare proxy',
    },
    'clinical_thresholds': {
        'Normal': {
            'bs_rate': '5–34/min',
            'iei_ms': '200–800ms',
            'action': 'No intervention needed',
        },
        'Hypomotility': {
            'bs_rate': '<2/min',
            'iei_ms': '>3000ms',
            'action': 'Assess for ileus, opioid effect, electrolyte imbalance',
            'urgent': 'Absent BS >6h → surgical review',
        },
        'Hypermotility': {
            'bs_rate': '>34/min',
            'iei_ms': '<200ms',
            'action': 'Stool chart, ORT, gastroenterology review if >48h',
        },
        'IBD_Active': {
            'bs_rate': '8–20/min',
            'iei_ms': '>1000ms',
            'action': 'IBD nurse alert, stool biomarker (calprotectin), urgent GI review',
            'crohn_signature': 'IEI >1000ms + high-pitch spectral component',
        },
        'Obstruction': {
            'bs_rate': 'Initially elevated, then absent',
            'spectral': 'High-pitched tinkling >600Hz',
            'action': '🚨 EMERGENCY — surgical referral, CT abdomen',
        },
    },
    'medverse_integration': {
        'stress_ans_model':   'Model 05 — ANS index correlates with gut motility',
        'cardiac_age_model':  'Model 04 — IBD chronic inflammation → cardiac age',
        'temperature_model':  'Model 17 — shared temperature signal',
        'nutrition_model':    'Model 18 — dietary triggers for IBD flare detection',
        'report_frequency':   'Every 30s sliding window (15s stride)',
        'daily_summary':      'BS rate trend × 24h + motility index plot',
        'alert_escalation': {
            'level_1': 'Motility index outlier → user notification',
            'level_2': 'IBD_Active pattern ≥2 windows → IBD nurse',
            'level_3': 'Obstruction pattern → 999/A&E referral (EMERGENCY)',
        },
    },
    'uncertainty': {
        'high_noise_flag':  'ACC RMS > 0.05 (motion artefact) → flag for review',
        'poor_acoustic':    'Spec RMS < 0.005 → request re-application of patch',
        'model_disagreement': 'BS-CNN vs Transformer > 2 class diff → manual auscultation',
    },
}

with open('medverse_gi_motility_config.json', 'w') as f:
    json.dump(config_gi, f, indent=2)

print("Saved:")
for fname in ['medverse_gi_bs_cnn.pt', 'medverse_gi_transformer.pt',
              'medverse_gi_multimodal.pt', 'medverse_gi_lgbm.pkl',
              'medverse_gi_scaler.pkl', 'medverse_gi_feat_cols.pkl',
              'medverse_gi_motility_config.json']:
    if os.path.exists(fname):
        print(f"  {fname:42s} {os.path.getsize(fname)/1024:7.1f} KB")


In [ ]:
def predict_gi_motility(audio_signal, acc_signal=None, ecg_signal=None,
                          temp_signal=None,
                          fs_audio=8000, fs_acc=100, fs_ecg=500,
                          window_sec=30, past_windows=None):
    """
    MedVerse vest real-time GI Motility / Bowel classification.

    audio_signal: numpy (N,) — mandatory abdominal microphone @ 8kHz
    acc_signal:   numpy (N,3) — optional abdominal ACC @ 100Hz
    ecg_signal:   numpy (N,) — optional ECG @ 500Hz
    temp_signal:  numpy (N,) — optional core temperature @ 1Hz
    past_windows: list of past GI class predictions (trend analysis)
    """
    start_t = time.time()

    # ── Step 1: Validate & normalise ─────────────────
    n_audio_min = int(window_sec * fs_audio)
    if len(audio_signal) < n_audio_min:
        return {'error': f'Need ≥{window_sec}s audio — '
                          f'got {len(audio_signal)/fs_audio:.0f}s'}

    audio   = audio_signal[:n_audio_min].astype(np.float32)
    audio_n = audio / (np.abs(audio).max() + 1e-8)

    # ── Step 2: Signal quality check ─────────────────
    rms_audio = float(np.sqrt(np.mean(audio_n**2)))
    quality   = ('GOOD'     if rms_audio > 0.008 else
                 'DEGRADED' if rms_audio > 0.003 else 'POOR')
    if quality == 'POOR':
        return {'error': 'Bowel sound signal too weak — '
                          'check abdominal patch contact on vest.'}

    # ── Step 3: Feature extraction ────────────────────
    bs_feats   = extract_bowel_sound_features(audio_n, fs=fs_audio)

    acc_3d = (acc_signal[:int(window_sec*fs_acc)]
               if acc_signal is not None
               else np.column_stack([
                   np.zeros(int(window_sec*fs_acc)),
                   np.zeros(int(window_sec*fs_acc)),
                   np.ones(int(window_sec*fs_acc)) * 0.008]))
    acc_feats  = extract_acc_gi_features(acc_3d, fs=fs_acc)

    rr_ints = np.array([800.0] * 20)   # default neutral HRV
    if ecg_signal is not None:
        ecg_arr = ecg_signal[:int(window_sec*fs_ecg)].astype(np.float32)
        sos     = sp_signal.butter(4, [0.5, min(40, fs_ecg/2-1)],
                                    btype='bandpass', fs=fs_ecg, output='sos')
        ecg_f   = sp_signal.sosfiltfilt(sos, ecg_arr)
        r_peaks, _ = sp_signal.find_peaks(
            ecg_f, height=np.percentile(ecg_f, 85),
            distance=int(0.3 * fs_ecg))
        if len(r_peaks) >= 4:
            rr_ints = np.diff(r_peaks) / fs_ecg * 1000

    hrv_feats  = extract_hrv_gi_features(rr_ints)

    temp_arr   = (temp_signal[:max(1, int(window_sec*1))]
                   if temp_signal is not None
                   else np.array([36.8], dtype=np.float32))
    temp_feats = {
        'temp_mean':  float(temp_arr.mean()),
        'temp_max':   float(temp_arr.max()),
        'temp_fever': float(temp_arr.max() > 37.5),
    }

    # ── Step 4: Classical model prediction ────────────
    all_feats = {}
    all_feats.update(bs_feats)
    all_feats.update(acc_feats)
    all_feats.update(hrv_feats)
    all_feats.update(temp_feats)

    feat_vec   = np.array([all_feats.get(f, 0.0) for f in gi_feat_cols],
                            dtype=np.float32).reshape(1, -1)
    feat_vec_s = scaler_gi.transform(feat_vec)
    lgbm_proba = gi_models['LightGBM'].predict_proba(feat_vec_s)[0]

    # ── Step 5: Deep model predictions ────────────────
    # Resample audio to model input length
    audio_model = sp_signal.resample(audio_n, int(SEG_SEC * FS_BS))
    audio_t     = torch.FloatTensor(audio_model).unsqueeze(0).unsqueeze(0).to(device)

    bs_cnn.eval(); gi_transformer.eval(); gi_mm_net.eval()
    with torch.no_grad():
        proba_cnn   = F.softmax(bs_cnn(audio_t),         dim=1).cpu().numpy()[0]
        proba_trans = F.softmax(gi_transformer(audio_t), dim=1).cpu().numpy()[0]

        # Multimodal — prepare all modalities
        acc_mm = torch.FloatTensor(
            sp_signal.resample(acc_3d[:,2], int(SEG_SEC*FS_ACC))
        ).unsqueeze(0).to(device)
        acc_full = torch.zeros(1, int(SEG_SEC*FS_ACC), 3).to(device)
        acc_full[:, :, 2] = acc_mm

        ecg_mm_arr = (ecg_signal[:int(window_sec*fs_ecg)]
                       if ecg_signal is not None
                       else np.zeros(int(window_sec*fs_ecg)))
        ecg_mm_arr = sp_signal.resample(
            ecg_mm_arr.astype(np.float32), int(SEG_SEC * FS_ECG))
        ecg_mm_n   = ecg_mm_arr / (np.abs(ecg_mm_arr).max() + 1e-8)
        ecg_t      = torch.FloatTensor(ecg_mm_n).unsqueeze(0).to(device)

        temp_t     = torch.FloatTensor([temp_arr.mean()]).unsqueeze(0).to(device)

        proba_mm   = F.softmax(
            gi_mm_net(audio_t.squeeze(1), acc_full, ecg_t, temp_t),
            dim=1).cpu().numpy()[0]

    # ── Step 6: Ensemble ──────────────────────────────
    # Weights: Transformer (0.38) > Multimodal (0.32) > CNN (0.18) > LGBM (0.12)
    ensemble_proba = (0.38 * proba_trans +
                      0.32 * proba_mm    +
                      0.18 * proba_cnn   +
                      0.12 * lgbm_proba)
    pred_class     = int(ensemble_proba.argmax())
    confidence     = float(ensemble_proba.max())

    # ── Step 7: Compute GI Motility Index (0–1) ───────
    gi_motility_idx = float(np.clip(
        0.35 * min(bs_feats.get('bs_event_rate', 0) / 34.0, 1.0) +
        0.25 * min(bs_feats.get('spec_mid_power', 0) * 3, 1.0) +
        0.20 * min(acc_feats.get('acc_motility_index', 0) * 50, 1.0) +
        0.20 * min(bs_feats.get('spec_high_power', 0) * 5, 1.0),
        0, 1))

    # ── Step 8: Trend / pattern analysis ─────────────
    obstruction_escalating = False
    ibd_flare_pattern      = False
    if past_windows is not None and len(past_windows) >= 4:
        recent = past_windows[-4:]
        # Obstruction: high-motility → sudden silence pattern
        obstruction_escalating = (
            any(r == 4 for r in recent[-2:]) or
            (any(r == 2 for r in recent[:2]) and
             any(r == 1 for r in recent[2:])))
        # IBD flare: ≥3 of last 4 windows = IBD_Active
        ibd_flare_pattern = (sum(1 for r in recent if r == 3) >= 3)

    # ── Step 9: Clinical risk & actions ───────────────
    clinical_actions = {
        0: {   # Normal
            'tier':    '🟢 NORMAL',
            'message': 'GI motility within normal range (5–34 BS/min)',
            'actions': [
                'No clinical action required',
                'Continue standard dietary habits',
                'Next assessment in 30 minutes (routine)',
            ],
            'urgent': False,
        },
        1: {   # Hypomotility
            'tier':    '🔵 HYPOMOTILITY',
            'message': 'Reduced/absent bowel sounds — possible ileus or constipation',
            'actions': [
                'Log stool frequency and type (Bristol Stool Scale)',
                'Review medications — opioids, anticholinergics, antidiarrhoea',
                'Encourage ambulation and hydration',
                'Abdominal assessment at next clinical contact',
                'If absent BS persisting >6h → URGENT surgical review',
            ],
            'urgent': bs_feats.get('bs_event_rate', 10) < 0.5,
        },
        2: {   # Hypermotility
            'tier':    '🟡 HYPERMOTILITY',
            'message': 'Hyperactive bowel sounds — possible diarrhoea or early obstruction',
            'actions': [
                'Stool frequency + consistency chart',
                'Oral rehydration therapy if diarrhoea suspected',
                'Review recent diet — fibre load, lactose, sorbitol',
                'If no improvement in 48h → GP/gastroenterology review',
                'Check for recent antibiotic use (C. diff risk)',
            ],
            'urgent': False,
        },
        3: {   # IBD Active
            'tier':    '🟠 IBD ACTIVE FLARE',
            'message': ('IBD acoustic signature detected '
                        '(IEI pattern consistent with Crohn\'s/UC flare)'),
            'actions': [
                '⚠️  IBD nurse / gastroenterologist alert triggered',
                'Faecal calprotectin test recommended (IBD activity marker)',
                'Record stool frequency, blood in stool, pain score',
                'Review current biologic/5-ASA therapy adherence',
                'Assess hydration status — IV fluids if severe',
                'Harvey-Bradshaw / partial Mayo score assessment',
                'Consider steroid rescue if not on current therapy',
            ],
            'urgent': ibd_flare_pattern,
        },
        4: {   # Obstruction
            'tier':    '🔴 POSSIBLE OBSTRUCTION',
            'message': 'High-pitched tinkling bowel sounds — mechanical obstruction suspected',
            'actions': [
                '🚨 EMERGENCY — contact 999 / A&E referral immediately',
                'DO NOT administer food, water, or oral medications',
                'IV access and fluid resuscitation',
                'Urgent AXR / CT abdomen with contrast',
                'Surgical team review — possible laparotomy',
                'Monitor vital signs every 15 minutes',
            ],
            'urgent': True,
        },
    }[pred_class]

    # Override urgency for escalating obstruction pattern
    if obstruction_escalating:
        clinical_actions['urgent']  = True
        clinical_actions['tier']   += ' ⚠️  ESCALATING'
        clinical_actions['actions'].insert(
            0, '🚨 Pattern shows escalating obstruction — EMERGENCY')

    inference_time = (time.time() - start_t) * 1000

    return {
        # ── Core outputs ──────────────────────────────
        'predicted_class':   GI_CLASSES[pred_class],
        'class_id':          pred_class,
        'confidence':        round(confidence, 4),
        'class_proba': {
            GI_CLASSES[i]: round(float(ensemble_proba[i]), 4)
            for i in range(N_GI_CLASSES)
        },
        # ── Motility index ────────────────────────────
        'gi_motility_index': round(gi_motility_idx, 4),
        'motility_tier':     ('Hypomotility' if gi_motility_idx < 0.3 else
                               'Normal'       if gi_motility_idx < 0.7 else
                               'Hypermotility'),
        # ── Key acoustic biomarkers ───────────────────
        'bowel_sound_snapshot': {
            'bs_rate_per_min': round(bs_feats.get('bs_event_rate', 0), 1),
            'iei_mean_ms':     round(bs_feats.get('bs_iei_mean_ms', 0), 0),
            'iei_gt1000ms':    round(bs_feats.get('bs_iei_gt1000ms', 0), 3),
            'spec_peak_hz':    round(bs_feats.get('spec_peak_hz', 0), 1),
            'spec_high_ratio': round(bs_feats.get('spec_high_ratio', 0), 4),
            'high_pitch_flag': bool(bs_feats.get('spec_high_power', 0) >
                                     bs_feats.get('spec_mid_power', 1) * 0.4),
        },
        # ── HRV gut-brain axis ────────────────────────
        'hrv_gutbrain': {
            'rmssd_ms':  round(hrv_feats.get('hrv_rmssd', 0), 1),
            'lf_hf':     round(hrv_feats.get('hrv_lf_hf', 0), 3),
            'mean_hr':   round(hrv_feats.get('hrv_mean_hr', 0), 1),
        },
        # ── Temperature ──────────────────────────────
        'temperature': {
            'core_temp_c': round(float(temp_arr.mean()), 2),
            'fever_flag':  bool(temp_arr.max() > 37.5),
        },
        # ── Trend analysis ────────────────────────────
        'obstruction_escalating':   obstruction_escalating,
        'ibd_flare_pattern':        ibd_flare_pattern,
        # ── Clinical decision ─────────────────────────
        'clinical_tier':    clinical_actions['tier'],
        'clinical_message': clinical_actions['message'],
        'clinical_actions': clinical_actions['actions'],
        'urgent_referral':  clinical_actions['urgent'],
        # ── Model breakdown ───────────────────────────
        'model_probabilities': {
            'transformer': {GI_CLASSES[i]: round(float(proba_trans[i]), 4)
                            for i in range(N_GI_CLASSES)},
            'multimodal':  {GI_CLASSES[i]: round(float(proba_mm[i]),    4)
                            for i in range(N_GI_CLASSES)},
            'bs_cnn':      {GI_CLASSES[i]: round(float(proba_cnn[i]),   4)
                            for i in range(N_GI_CLASSES)},
            'lightgbm':    {GI_CLASSES[i]: round(float(lgbm_proba[i]),  4)
                            for i in range(N_GI_CLASSES)},
        },
        # ── Signal quality ────────────────────────────
        'signal_quality':         quality,
        'modalities_used':        sum([True,
                                       acc_signal  is not None,
                                       ecg_signal  is not None,
                                       temp_signal is not None]),
        # ── MedVerse integration ──────────────────────
        'medverse': {
            'alert_triggered':       pred_class in [1, 2, 3, 4],
            'emergency_alert':       pred_class == 4 or obstruction_escalating,
            'ibd_nurse_alert':       pred_class == 3 or ibd_flare_pattern,
            'notify_user':           pred_class in [1, 2, 3],
            'notify_gp':             pred_class == 3 and ibd_flare_pattern,
            'activate_stress_model': True,   # Model 05 — gut-brain axis
            'activate_temp_model':   True,   # Model 17 — fever monitoring
            'log_motility_trend':    True,
            'next_window_sec':       (15 if pred_class == 4 else
                                      30 if pred_class == 3 else 30),
        },
        'inference_time_ms': round(inference_time, 1),
        'timestamp':         time.strftime('%Y-%m-%dT%H:%M:%S'),
    }

# ── Demo inference ────────────────────────────────────────
print("="*65)
print("MedVerse — GI Motility / Bowel: Demo Inference")
print("="*65)

for test_cls, label in GI_CLASSES.items():
    demo_rec = generate_gi_segment(test_cls, subject_id=99,
                                    seg_sec=SEG_SEC, seed=9000+test_cls)
    result   = predict_gi_motility(
        audio_signal = demo_rec['audio'],
        acc_signal   = demo_rec['acc'],
        ecg_signal   = demo_rec['ecg'],
        temp_signal  = demo_rec['temp'],
        fs_audio=FS_BS, fs_acc=FS_ACC, fs_ecg=FS_ECG,
    )
    print(f"\n{'─'*60}")
    print(f"  Ground Truth:      {label}")
    print(f"  Prediction:        {result['predicted_class']} "
          f"(conf={result['confidence']:.3f})")
    print(f"  Class Proba:       {result['class_proba']}")
    print(f"  Motility Index:    {result['gi_motility_index']:.4f} "
          f"→ {result['motility_tier']}")
    print(f"  Clinical Tier:     {result['clinical_tier']}")
    print(f"\n  Bowel Sound Snapshot:")
    for k, v in result['bowel_sound_snapshot'].items():
        print(f"    {k:22s}: {v}")
    print(f"\n  HRV (gut-brain):   {result['hrv_gutbrain']}")
    print(f"  Temperature:       {result['temperature']}")
    print(f"  Signal Quality:    {result['signal_quality']}")
    print(f"  Modalities Used:   {result['modalities_used']}/4")
    print(f"\n  Clinical Actions:")
    for a in result['clinical_actions']:
        print(f"    • {a}")
    print(f"  Urgent Referral:   {result['urgent_referral']}")
    print(f"  MedVerse Flags:    {result['medverse']}")
    print(f"  Inference Time:    {result['inference_time_ms']}ms")


In [ ]:
print("\n" + "="*65)
print("✅  GI Motility / Bowel Classifier — Model 16 Complete")
print("="*65)
print(f"  Datasets:        WESAD-matched synthetic (WESAD protocol)")
print(f"                   IBD Smart T-Shirt ref (24 IBD+21 healthy, 281h)")
print(f"                   Bowel Sound AI analytics (PMC 2025)")
print(f"  Classes:         Normal | Hypomotility | Hypermotility |")
print(f"                   IBD_Active | Obstruction")
print(f"  Models:          BowelSoundCNN (multi-scale raw audio)")
print(f"                   GIPatchTransformer (Wav2Vec2-style, F1 ~83%)")
print(f"                   GIMultimodalNet (Audio+ACC+ECG+Temp fusion)")
print(f"                   LightGBM (acoustic + HRV features)")
print(f"  LOSO Val:        {np.mean(loso_accs_gi):.3f} ± {np.std(loso_accs_gi):.3f} (subject-independent)")
print(f"  Key Biomarker:   IEI=1232ms → Crohn's | IEI=511ms → IBS (PMC 2025)")
print(f"  High-Pitch Flag: Spectral >600Hz → Obstruction / IBD stricture")
print(f"  Gut-Brain Axis:  HRV RMSSD < 20ms → active inflammation proxy")
print(f"  Emergency Path:  Obstruction pattern → 999/A&E in <60s")
print(f"  MedVerse:        8kHz abdominal mic + 3-axis ACC vest patch,")
print(f"                   30s windows, integrates Model 05+17+18")
print("="*65)